In [1]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import hashlib

In [2]:
# =========================
# Configurações principais
# =========================

SEED = 42

GOAT_SOURCE = Path(r"D:\brats_goat_experiments\data\raw_oficial_training_validation_set\MICCAI2024-BraTS-GoAT-TrainingData-With-GroundTruth")

SSA_SOURCE = Path(r"D:\brats_goat_experiments\data\raw_outside_test_set\BraTS-Africa")

PED_SOURCE = Path(r"D:\brats_goat_experiments\data\raw_outside_test_set\ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData")

SPLIT_ROOT = Path(r"D:\brats_goat_experiments\data\split")

TRAIN_DEST = SPLIT_ROOT / "train"
VAL_DEST = SPLIT_ROOT / "val"
TEST_ROOT = SPLIT_ROOT / "test"

TEST_GLI_MET_MEN_DEST = TEST_ROOT / "test_GLI-MET-MEN"
SSA_DEST = TEST_ROOT / "SSA"
PED_DEST = TEST_ROOT / "PED"

# Se True, apaga as pastas de destino antes de copiar novamente
CLEAR_DESTINATION = False

In [3]:
# =========================
# Funções auxiliares
# =========================

def validate_path(path: Path, name: str):
    if not path.exists():
        raise FileNotFoundError(f"{name} não encontrado: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} não é um diretório: {path}")


def prepare_dir(path: Path, clear: bool = False):
    if clear and path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def copy_folder(src: Path, dst: Path):
    if dst.exists():
        raise FileExistsError(f"Destino já existe: {dst}")
    shutil.copytree(src, dst)


def copy_all_elements(src_root: Path, dst_root: Path):
    elements = sorted(src_root.iterdir())

    for element in tqdm(elements, desc=f"Copiando {src_root.name}"):
        dst = dst_root / element.name

        if dst.exists():
            raise FileExistsError(f"Destino já existe: {dst}")

        if element.is_dir():
            shutil.copytree(element, dst)
        else:
            shutil.copy2(element, dst)

In [4]:
# =========================
# Validação das origens
# =========================

validate_path(GOAT_SOURCE, "Dataset BraTS-GoAT")
validate_path(SSA_SOURCE, "Dataset SSA")
validate_path(PED_SOURCE, "Dataset PED")

print("Todas as pastas de origem foram encontradas.")

Todas as pastas de origem foram encontradas.


In [5]:
# =========================
# Preparação das pastas
# =========================

prepare_dir(TRAIN_DEST, clear=CLEAR_DESTINATION)
prepare_dir(VAL_DEST, clear=CLEAR_DESTINATION)
prepare_dir(TEST_ROOT, clear=CLEAR_DESTINATION)

prepare_dir(TEST_GLI_MET_MEN_DEST, clear=CLEAR_DESTINATION)
prepare_dir(SSA_DEST, clear=CLEAR_DESTINATION)
prepare_dir(PED_DEST, clear=CLEAR_DESTINATION)

print("Pastas de destino preparadas.")

Pastas de destino preparadas.


In [6]:
# =========================
# Listagem dos pacientes GoAT
# =========================

patients = sorted([p for p in GOAT_SOURCE.iterdir() if p.is_dir()])

if len(patients) == 0:
    raise ValueError(f"Nenhuma pasta de paciente encontrada em: {GOAT_SOURCE}")

print(f"Total de pacientes encontrados no BraTS-GoAT: {len(patients)}")

Total de pacientes encontrados no BraTS-GoAT: 1351


In [7]:
# =========================
# Divisão 70%, 15%, 15%
# =========================

random.seed(SEED)
random.shuffle(patients)

n_total = len(patients)

n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
n_test = n_total - n_train - n_val

train_patients = patients[:n_train]
val_patients = patients[n_train:n_train + n_val]
test_patients = patients[n_train + n_val:]

print(f"Treino:    {len(train_patients)} pacientes")
print(f"Validação: {len(val_patients)} pacientes")
print(f"Teste:     {len(test_patients)} pacientes")
print(f"Total:     {len(train_patients) + len(val_patients) + len(test_patients)} pacientes")

Treino:    945 pacientes
Validação: 202 pacientes
Teste:     204 pacientes
Total:     1351 pacientes


In [8]:
# =========================
# Cópia do BraTS-GoAT
# =========================

for patient in tqdm(train_patients, desc="Copiando treino"):
    copy_folder(patient, TRAIN_DEST / patient.name)

for patient in tqdm(val_patients, desc="Copiando validação"):
    copy_folder(patient, VAL_DEST / patient.name)

for patient in tqdm(test_patients, desc="Copiando teste GLI-MET-MEN"):
    copy_folder(patient, TEST_GLI_MET_MEN_DEST / patient.name)

print("Divisão do BraTS-GoAT concluída.")

Copiando teste GLI-MET-MEN: 100%|██████████| 204/204 [00:07<00:00, 28.04it/s]

Divisão do BraTS-GoAT concluída.


In [9]:
# =========================
# Cópia dos datasets externos
# =========================

copy_all_elements(SSA_SOURCE, SSA_DEST)
copy_all_elements(PED_SOURCE, PED_DEST)

print("Datasets externos copiados para o conjunto de teste.")

Copiando BraTS-Africa: 100%|██████████| 146/146 [00:05<00:00, 28.58it/s]
Copiando ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData: 100%|██████████| 99/99 [00:26<00:00,  3.77it/s]

Datasets externos copiados para o conjunto de teste.


In [10]:
# =========================
# Resumo final em DataFrame
# =========================

def count_dirs(path: Path):
    return len([p for p in path.iterdir() if p.is_dir()])


summary_data = [
    {
        "dataset": "BraTS-GoAT",
        "split": "train",
        "path": str(TRAIN_DEST),
        "quantidade_pacientes": count_dirs(TRAIN_DEST)
    },
    {
        "dataset": "BraTS-GoAT",
        "split": "val",
        "path": str(VAL_DEST),
        "quantidade_pacientes": count_dirs(VAL_DEST)
    },
    {
        "dataset": "BraTS-GoAT",
        "split": "test_GLI-MET-MEN",
        "path": str(TEST_GLI_MET_MEN_DEST),
        "quantidade_pacientes": count_dirs(TEST_GLI_MET_MEN_DEST)
    },
    {
        "dataset": "BraTS-Africa",
        "split": "test_SSA",
        "path": str(SSA_DEST),
        "quantidade_pacientes": count_dirs(SSA_DEST)
    },
    {
        "dataset": "BraTS-PED",
        "split": "test_PED",
        "path": str(PED_DEST),
        "quantidade_pacientes": count_dirs(PED_DEST)
    }
]

df_summary = pd.DataFrame(summary_data)

df_summary

,dataset,split,path,quantidade_pacientes
0,BraTS-GoAT,train,D:\brats_goat_experiments\data\split\train,945
1,BraTS-GoAT,val,D:\brats_goat_experiments\data\split\val,202
2,BraTS-GoAT,test_GLI-MET-MEN,D:\brats_goat_experiments\data\split\test\test...,204
3,BraTS-Africa,test_SSA,D:\brats_goat_experiments\data\split\test\SSA,146
4,BraTS-PED,test_PED,D:\brats_goat_experiments\data\split\test\PED,99


In [11]:
# =========================
# Verificação de duplicatas por hash entre train, val e test
# =========================

def hash_file(file_path: Path, chunk_size: int = 1024 * 1024):
    hasher = hashlib.sha256()

    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)

    return hasher.hexdigest()


def hash_patient_folder(patient_path: Path):
    """
    Gera um hash único para a pasta do paciente.
    Considera nomes relativos dos arquivos + conteúdo dos arquivos.
    """
    hasher = hashlib.sha256()

    files = sorted([p for p in patient_path.rglob("*") if p.is_file()])

    for file_path in files:
        relative_path = file_path.relative_to(patient_path).as_posix()
        file_hash = hash_file(file_path)

        hasher.update(relative_path.encode("utf-8"))
        hasher.update(file_hash.encode("utf-8"))

    return hasher.hexdigest()


def collect_patient_hashes(split_name: str, split_path: Path):
    records = []

    patient_dirs = sorted([p for p in split_path.iterdir() if p.is_dir()])

    for patient_dir in tqdm(patient_dirs, desc=f"Calculando hashes: {split_name}"):
        records.append({
            "split": split_name,
            "patient_id": patient_dir.name,
            "path": str(patient_dir),
            "folder_hash": hash_patient_folder(patient_dir)
        })

    return records


hash_records = []

hash_records.extend(collect_patient_hashes("train", TRAIN_DEST))
hash_records.extend(collect_patient_hashes("val", VAL_DEST))
hash_records.extend(collect_patient_hashes("test_GLI-MET-MEN", TEST_GLI_MET_MEN_DEST))
hash_records.extend(collect_patient_hashes("test_SSA", SSA_DEST))
hash_records.extend(collect_patient_hashes("test_PED", PED_DEST))

df_hashes = pd.DataFrame(hash_records)

# Identifica hashes repetidos
df_duplicates = (
    df_hashes[df_hashes.duplicated(subset=["folder_hash"], keep=False)]
    .sort_values(["folder_hash", "split", "patient_id"])
    .reset_index(drop=True)
)

# Verifica se a duplicata aparece em mais de um split
if not df_duplicates.empty:
    df_leakage = (
        df_duplicates
        .groupby("folder_hash")
        .filter(lambda x: x["split"].nunique() > 1)
        .sort_values(["folder_hash", "split", "patient_id"])
        .reset_index(drop=True)
    )
else:
    df_leakage = pd.DataFrame(columns=df_hashes.columns)

# Resumo em DataFrame
df_leakage_summary = pd.DataFrame([
    {
        "verificacao": "Duplicatas por hash entre splits",
        "total_pacientes_analisados": len(df_hashes),
        "total_hashes_duplicados": df_duplicates["folder_hash"].nunique() if not df_duplicates.empty else 0,
        "total_casos_com_possivel_leakage": df_leakage["folder_hash"].nunique() if not df_leakage.empty else 0,
        "status": "OK - nenhuma duplicata entre splits encontrada" if df_leakage.empty else "ATENÇÃO - possível data leakage encontrado"
    }
])

display(df_leakage_summary)

# Exibe detalhes apenas se houver possível vazamento
if not df_leakage.empty:
    display(df_leakage)
else:
    print("Nenhuma duplicata exata encontrada entre treino, validação e teste.")

Calculando hashes: test_PED: 100%|██████████| 99/99 [00:02<00:00, 38.45it/s]


,verificacao,total_pacientes_analisados,total_hashes_duplicados,total_casos_com_possivel_leakage,status
0,Duplicatas por hash entre splits,1596,0,0,OK - nenhuma duplicata entre splits encontrada


Nenhuma duplicata exata encontrada entre treino, validação e teste.
